# Reviewed remediation
Start with the separate [read-only preview](docs/QUICKSTART.md). This notebook plans, rehearses and can change data. Use a disposable PostgreSQL copy first; see [operator safeguards](docs/SAFETY.md).

**Do not use Run All while connected. Defaults are disconnected and commit is off.**

In [ ]:
from pathlib import Path
from datetime import date
from getpass import getpass
from remediation.runner import Config, connect, preview, export_preview, build_plan, save_plan, apply_plan, rollback_run, discover_event_scope
from remediation.reporting import apply_and_report, generate_committed_report
RUN_DATABASE = False
COMMIT_CHANGES = False
conn = None
plan = None
print("No database connection has been made.")
from remediation.reporting import generate_audit_only_report

## 1. Choose the database and scope
Use approved schemas, matching CDM version, read-only credentials and private output outside the project. Keep TLS verification on. Confirm discovered scope; a site label is not a row filter.

In [ ]:
cfg = Config(
    cdm_schema="omopcdm",
    vocabulary_schema="vocabulary",
    tables=("condition_occurrence",),  # Bootstrap only; discovery below selects all visible event tables.
    cdm_version="5.4",  # Set 5.4 or 5.5 to match CDM_SOURCE.cdm_version.
    as_of=date.today().isoformat(),
)
private_output = Path.home() / "private_remediation_results"  # Change to site-approved storage.
if RUN_DATABASE:
    conn = connect(host=input("Database host: "), port=int(input("Port [5432]: ") or "5432"),
                   dbname=input("Database name: "), user=input("Database user: "),
                   password=getpass("Password (not saved): "), sslmode="verify-full",
                   sslrootcert=input("CA certificate path (blank for system defaults): ") or None)
    cfg, coverage = discover_event_scope(conn, cfg)
    print("Selected available event tables:", cfg.tables)
    for entry in coverage:
        print(entry["table"], entry["column"], entry["state"], entry["policy"])
else:
    print("Connection skipped. Set RUN_DATABASE only in your approved environment.")

## 2. Preview with no writes
Review `preview.csv`. `ELIGIBLE_FOR_REVIEW` is not approval or proof of equivalence.

In [ ]:
if RUN_DATABASE and conn is not None:
    records = preview(conn, cfg)
    export_preview(records, private_output / "preview.csv")
    from collections import Counter
    print(Counter(r["status"] for r in records))
    print("Private aggregate preview saved.")

## 3. Freeze the exact changes
Confirm scope, vocabulary, counts and meaning. The private plan holds row IDs; do not edit it. Narrow scope and rebuild if needed. Zero rows: read the audit-only PDF and skip rehearsal/apply.

In [ ]:
if RUN_DATABASE and conn is not None:
    plan = build_plan(conn, cfg)
    plan_path = private_output / f"plan_{plan['run_id']}.json"
    save_plan(plan, plan_path)
    print("Rows:", len(plan["rows"]))
    print("Review plan hash:", plan["plan_hash"])
    print("Plan saved. No changes made.")
    if not plan["rows"]:
        print(generate_audit_only_report(plan, private_output / f"audit_only_{plan['run_id']}.pdf"))
        print("No apply: skip rehearsal/commit; review audit findings and exclusions.")

## 4. Rehearse and roll back
Follow the [operator guide](docs/SAFETY.md) to reconnect with the authorized writer while retaining the reviewed plan/configuration. Paste the reviewed hash. The rehearsal takes locks; `committed` must be `False`.

In [ ]:
approved_plan_hash = ""  # Paste the reviewed plan hash, not automatic assignment.
if RUN_DATABASE and conn is not None and plan is not None and plan["rows"] and approved_plan_hash:
    rehearsal = apply_plan(conn, cfg, plan, approval_hash=approved_plan_hash)
    print(rehearsal)  # committed must be False.
else:
    print("Rehearsal not requested.")

## 5. Commit only with separate approval
After VM acceptance, pause competing ETL and verify a restorable backup. Set the commit controls only for the approved run. Changes and journal commit together. If the commit response is lost, reconcile its run ID before any retry.

In [ ]:
COMMIT_CHANGES = False
backup_reference = ""
maintenance_confirmed = False
if COMMIT_CHANGES:
    if not RUN_DATABASE or conn is None or plan is None:
        raise RuntimeError("Connection and reviewed plan required")
    result = apply_and_report(conn, cfg, plan,
                        private_output / f"committed_{plan['run_id']}.pdf",
                        commit=True, approval_hash=approved_plan_hash,
                        backup_reference=backup_reference, maintenance_confirmed=maintenance_confirmed)
    print(result)
    if result.get("report_status") == "FAILED_AFTER_COMMIT":
        print("DATA ALREADY COMMITTED. Retry only report generation from run_id; do not re-apply.")
else:
    print("Commit disabled.")

## Recover only the PDF if reporting failed
`FAILED_AFTER_COMMIT` means data are already committed. Never repeat apply. Use [journal-only recovery](docs/REPORTING.md) and a new private filename.

## 6. Verify and close
Inspect the after-preview and journal/report. Check affected downstream cohorts and derived data separately. Keep private outputs out of Git.

In [ ]:
if RUN_DATABASE and conn is not None:
    export_preview(preview(conn, cfg), private_output / "after_preview.csv")
    conn.close()
    conn = None
    print("Connection closed.")

## Optional reversal of a confirmed run
Never globally swap new IDs back to old IDs. Follow the [journal-based reversal](docs/SAFETY.md) with backup and maintenance approval: rehearse first, then separately authorize commit. Later row edits may block reversal; restored IDs can be outdated.